# All 13 Machine Learning Algorithms on Loan Default Dataset
---
## Algorithms Covered:
1. Linear Regression
2. Logistic Regression
3. Decision Tree
4. Random Forest
5. Support Vector Machine (SVM)
6. K-Nearest Neighbors (KNN)
7. Naive Bayes
8. Gradient Boosting
9. XGBoost
10. AdaBoost
11. K-Means Clustering
12. DBSCAN
13. Ensemble Learning (Voting Classifier)

---
## Step 1: Import Libraries
---

In [ ]:
# Install required libraries if needed
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
# ============================================================
# IMPORT ALL REQUIRED LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, mean_squared_error, mean_absolute_error, r2_score,
                             silhouette_score)

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, VotingClassifier, BaggingClassifier,
                              StackingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.cluster import KMeans, DBSCAN

print("✅ All libraries imported successfully!")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ============================================================
# LOAD THE DATASET
# ============================================================
df = pd.read_csv('loan_default_dataset.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*60)
print("First 5 Rows:")
df.head()

In [ ]:
# ============================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
print("="*60)
print("DATASET INFO")
print("="*60)
print(df.info())
print("\n" + "="*60)
print("STATISTICAL SUMMARY")
print("="*60)
df.describe()

In [ ]:
# Check for missing values
print("="*60)
print("MISSING VALUES")
print("="*60)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found!")

print("\n" + "="*60)
print("COLUMN DATA TYPES")
print("="*60)
print(df.dtypes)

In [ ]:
# Check all column names
print("Column Names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} -> dtype: {df[col].dtype} -> unique: {df[col].nunique()}")

In [ ]:
# ============================================================
# VISUALIZATIONS - EDA
# ============================================================

# Identify the target column (adjust if needed)
# Common target names: 'Default', 'default', 'Loan_Status', 'loan_default', 'LoanDefault'
target_col = None
possible_targets = ['Default', 'default', 'Loan_Status', 'loan_default', 'LoanDefault', 
                    'loan_status', 'is_default', 'TARGET', 'target', 'Status']

for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    # If no common target found, use the last column
    target_col = df.columns[-1]
    print(f"⚠️ Auto-selected last column as target: '{target_col}'")
else:
    print(f"✅ Target column identified: '{target_col}'")

print(f"\nTarget Distribution:\n{df[target_col].value_counts()}")

In [ ]:
# Target Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title(f'Target Distribution: {target_col}', fontsize=14, fontweight='bold')
axes[0].set_xlabel(target_col)
axes[0].set_ylabel('Count')

# Pie chart
df[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                   colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title(f'Target Proportion: {target_col}', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap (for numeric columns only)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    plt.figure(figsize=(14, 10))
    corr_matrix = df[numeric_cols].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                fmt='.2f', linewidths=0.5, square=True)
    plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation heatmap.")

In [ ]:
# Distribution of numeric features
numeric_features = [col for col in numeric_cols if col != target_col]

if len(numeric_features) > 0:
    n_cols = 3
    n_rows = (len(numeric_features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes.flatten()
    
    for i, col in enumerate(numeric_features):
        if i < len(axes):
            axes[i].hist(df[col].dropna(), bins=30, color='#3498db', edgecolor='black', alpha=0.7)
            axes[i].set_title(f'Distribution of {col}', fontsize=10, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequency')
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# DATA PREPROCESSING
# ============================================================

# Make a copy
df_processed = df.copy()

# Handle missing values
# Fill numeric columns with median
for col in df_processed.select_dtypes(include=[np.number]).columns:
    df_processed[col].fillna(df_processed[col].median(), inplace=True)

# Fill categorical columns with mode
for col in df_processed.select_dtypes(include=['object']).columns:
    df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

# Encode categorical variables
label_encoders = {}
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\n✅ Preprocessing complete! Shape: {df_processed.shape}")
print(f"\nAll columns are now numeric:")
print(df_processed.dtypes)

In [ ]:
# ============================================================
# DEFINE FEATURES (X) AND TARGET (y)
# ============================================================

# Drop any ID-like columns
id_cols = [col for col in df_processed.columns if 'id' in col.lower() or 'unnamed' in col.lower()]
if id_cols:
    print(f"Dropping ID-like columns: {id_cols}")
    df_processed.drop(columns=id_cols, inplace=True)

X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")
print(f"\nTarget unique values: {sorted(y.unique())}")

In [ ]:
# ============================================================
# TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

print(f"Training set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Testing set:  X_test={X_test.shape}, y_test={y_test.shape}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTesting target distribution:\n{y_test.value_counts()}")

In [ ]:
# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Feature scaling complete!")
print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled testing set shape: {X_test_scaled.shape}")

In [ ]:
# ============================================================
# HELPER FUNCTION FOR CLASSIFICATION EVALUATION
# ============================================================

def evaluate_classifier(model_name, y_true, y_pred, y_prob=None):
    """Comprehensive evaluation of a classification model."""
    print("=" * 60)
    print(f"📊 {model_name} - EVALUATION RESULTS")
    print("=" * 60)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    if y_prob is not None:
        try:
            if len(np.unique(y_true)) == 2:
                auc = roc_auc_score(y_true, y_prob)
                print(f"AUC-ROC:   {auc:.4f}")
        except:
            pass
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    # Confusion Matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=sorted(np.unique(y_true)),
                yticklabels=sorted(np.unique(y_true)))
    axes[0].set_title(f'{model_name} - Confusion Matrix', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    
    # ROC Curve (for binary classification)
    if y_prob is not None and len(np.unique(y_true)) == 2:
        try:
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auc_val = roc_auc_score(y_true, y_prob)
            axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC={auc_val:.4f})')
            axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].set_title(f'{model_name} - ROC Curve', fontsize=12, fontweight='bold')
            axes[1].legend(loc='lower right')
            axes[1].grid(True, alpha=0.3)
        except:
            axes[1].text(0.5, 0.5, 'ROC not available', ha='center', va='center', fontsize=14)
            axes[1].set_title(f'{model_name} - ROC Curve', fontsize=12, fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, 'ROC for binary\nclassification only', 
                     ha='center', va='center', fontsize=14)
        axes[1].set_title(f'{model_name} - ROC Curve', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 
            'Recall': rec, 'F1-Score': f1}

In [ ]:
# Store all results for final comparison
all_results = []

---
## 🔷 Algorithm 1: LINEAR REGRESSION
---

In [ ]:
# ============================================================
# ALGORITHM 1: LINEAR REGRESSION
# ============================================================
print("🔷 ALGORITHM 1: LINEAR REGRESSION")
print("Note: Linear Regression is used for regression tasks.")
print("We apply it here and convert predictions for classification.")
print("="*60)

# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions (continuous)
y_pred_lr_continuous = lr_model.predict(X_test_scaled)

# Regression Metrics
print("\n📈 Regression Metrics:")
print(f"Mean Squared Error (MSE):  {mean_squared_error(y_test, y_pred_lr_continuous):.4f}")
print(f"Root MSE (RMSE):           {np.sqrt(mean_squared_error(y_test, y_pred_lr_continuous)):.4f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred_lr_continuous):.4f}")
print(f"R² Score:                  {r2_score(y_test, y_pred_lr_continuous):.4f}")

# Convert to classification (threshold = 0.5)
y_pred_lr_class = (y_pred_lr_continuous >= 0.5).astype(int)

print("\n📊 Classification Metrics (threshold=0.5):")
result_lr = evaluate_classifier("Linear Regression", y_test, y_pred_lr_class, y_pred_lr_continuous)
all_results.append(result_lr)

In [ ]:
# Linear Regression - Additional Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_lr_continuous, alpha=0.5, color='#3498db', s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Linear Regression: Actual vs Predicted', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residuals Distribution
residuals = y_test - y_pred_lr_continuous
axes[1].hist(residuals, bins=30, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Linear Regression: Residuals Distribution', fontsize=12, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Feature Coefficients
coeff_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})
coeff_df = coeff_df.reindex(coeff_df['Coefficient'].abs().sort_values(ascending=False).index)
print("\nTop Feature Coefficients:")
print(coeff_df.to_string(index=False))

---
## 🔷 Algorithm 2: LOGISTIC REGRESSION
---

In [ ]:
# ============================================================
# ALGORITHM 2: LOGISTIC REGRESSION
# ============================================================
print("🔷 ALGORITHM 2: LOGISTIC REGRESSION")
print("="*60)

# Train Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_log = evaluate_classifier("Logistic Regression", y_test, y_pred_log, y_prob_log)
all_results.append(result_log)

In [ ]:
# Logistic Regression - Feature Importance
if len(np.unique(y)) == 2:
    coeff_log = pd.DataFrame({'Feature': X.columns, 'Coefficient': log_reg.coef_[0]})
    coeff_log['Abs_Coefficient'] = coeff_log['Coefficient'].abs()
    coeff_log = coeff_log.sort_values('Abs_Coefficient', ascending=True)
    
    plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coeff_log['Coefficient']]
    plt.barh(coeff_log['Feature'], coeff_log['Coefficient'], color=colors, edgecolor='black')
    plt.xlabel('Coefficient Value')
    plt.title('Logistic Regression - Feature Coefficients', fontsize=14, fontweight='bold')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

---
## 🔷 Algorithm 3: DECISION TREE
---

In [ ]:
# ============================================================
# ALGORITHM 3: DECISION TREE
# ============================================================
print("🔷 ALGORITHM 3: DECISION TREE")
print("="*60)

# Train Decision Tree Model
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=5,
                                   min_samples_leaf=2)
dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_dt = evaluate_classifier("Decision Tree", y_test, y_pred_dt, y_prob_dt)
all_results.append(result_dt)

In [ ]:
# Decision Tree - Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Tree Plot (limited depth for readability)
plt.figure(figsize=(20, 10))
plot_tree(dt_model, max_depth=3, feature_names=X.columns.tolist(), 
          filled=True, rounded=True, fontsize=8, proportion=True)
plt.title('Decision Tree (max_depth=3 for visualization)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Importance
feat_imp_dt = pd.DataFrame({'Feature': X.columns, 'Importance': dt_model.feature_importances_})
feat_imp_dt = feat_imp_dt.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
plt.barh(feat_imp_dt['Feature'], feat_imp_dt['Importance'], color='#f39c12', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('Decision Tree - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🔷 Algorithm 4: RANDOM FOREST
---

In [ ]:
# ============================================================
# ALGORITHM 4: RANDOM FOREST
# ============================================================
print("🔷 ALGORITHM 4: RANDOM FOREST")
print("="*60)

# Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15,
                                   min_samples_split=5, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_rf = evaluate_classifier("Random Forest", y_test, y_pred_rf, y_prob_rf)
all_results.append(result_rf)

In [ ]:
# Random Forest - Feature Importance
feat_imp_rf = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
feat_imp_rf = feat_imp_rf.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
plt.barh(feat_imp_rf['Feature'], feat_imp_rf['Importance'], color='#27ae60', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 10 Important Features:")
print(feat_imp_rf.sort_values('Importance', ascending=False).head(10).to_string(index=False))

---
## 🔷 Algorithm 5: SUPPORT VECTOR MACHINE (SVM)
---

In [ ]:
# ============================================================
# ALGORITHM 5: SUPPORT VECTOR MACHINE (SVM)
# ============================================================
print("🔷 ALGORITHM 5: SUPPORT VECTOR MACHINE (SVM)")
print("="*60)

# For large datasets, use a subset for SVM (it's computationally expensive)
max_svm_samples = 10000
if len(X_train_scaled) > max_svm_samples:
    print(f"⚠️ Using subset of {max_svm_samples} samples for SVM (original: {len(X_train_scaled)})")
    indices = np.random.RandomState(42).choice(len(X_train_scaled), max_svm_samples, replace=False)
    X_train_svm = X_train_scaled[indices]
    y_train_svm = y_train.iloc[indices]
else:
    X_train_svm = X_train_scaled
    y_train_svm = y_train

# Train SVM Model
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train_svm, y_train_svm)

# Predictions
y_pred_svm = svm_model.predict(X_test_scaled)
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_svm = evaluate_classifier("SVM (RBF Kernel)", y_test, y_pred_svm, y_prob_svm)
all_results.append(result_svm)

---
## 🔷 Algorithm 6: K-NEAREST NEIGHBORS (KNN)
---

In [ ]:
# ============================================================
# ALGORITHM 6: K-NEAREST NEIGHBORS (KNN)
# ============================================================
print("🔷 ALGORITHM 6: K-NEAREST NEIGHBORS (KNN)")
print("="*60)

# Find optimal K
k_range = range(3, 21, 2)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn_temp.fit(X_train_scaled, y_train)
    score = knn_temp.score(X_test_scaled, y_test)
    k_scores.append(score)
    
best_k = list(k_range)[np.argmax(k_scores)]
print(f"Best K = {best_k} with accuracy = {max(k_scores):.4f}")

# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(list(k_range), k_scores, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='r', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN - K Value vs Accuracy', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train KNN with best K
knn_model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_knn = evaluate_classifier(f"KNN (K={best_k})", y_test, y_pred_knn, y_prob_knn)
all_results.append(result_knn)

---
## 🔷 Algorithm 7: NAIVE BAYES
---

In [ ]:
# ============================================================
# ALGORITHM 7: NAIVE BAYES (Gaussian)
# ============================================================
print("🔷 ALGORITHM 7: NAIVE BAYES (Gaussian)")
print("="*60)

# Train Naive Bayes Model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_nb = nb_model.predict(X_test_scaled)
y_prob_nb = nb_model.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_nb = evaluate_classifier("Naive Bayes", y_test, y_pred_nb, y_prob_nb)
all_results.append(result_nb)

---
## 🔷 Algorithm 8: GRADIENT BOOSTING
---

In [ ]:
# ============================================================
# ALGORITHM 8: GRADIENT BOOSTING
# ============================================================
print("🔷 ALGORITHM 8: GRADIENT BOOSTING")
print("="*60)

# Train Gradient Boosting Model
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, 
                                      max_depth=5, random_state=42,
                                      min_samples_split=5, min_samples_leaf=2)
gb_model.fit(X_train, y_train)

# Predictions
y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_gb = evaluate_classifier("Gradient Boosting", y_test, y_pred_gb, y_prob_gb)
all_results.append(result_gb)

In [ ]:
# Gradient Boosting - Feature Importance
feat_imp_gb = pd.DataFrame({'Feature': X.columns, 'Importance': gb_model.feature_importances_})
feat_imp_gb = feat_imp_gb.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
plt.barh(feat_imp_gb['Feature'], feat_imp_gb['Importance'], color='#9b59b6', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('Gradient Boosting - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🔷 Algorithm 9: XGBOOST
---

In [ ]:
# ============================================================
# ALGORITHM 9: XGBOOST
# ============================================================
print("🔷 ALGORITHM 9: XGBOOST")
print("="*60)

# Train XGBoost Model
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, 
                           random_state=42, use_label_encoder=False, 
                           eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_xgb = evaluate_classifier("XGBoost", y_test, y_pred_xgb, y_prob_xgb)
all_results.append(result_xgb)

In [ ]:
# XGBoost - Feature Importance
feat_imp_xgb = pd.DataFrame({'Feature': X.columns, 'Importance': xgb_model.feature_importances_})
feat_imp_xgb = feat_imp_xgb.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
plt.barh(feat_imp_xgb['Feature'], feat_imp_xgb['Importance'], color='#e67e22', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('XGBoost - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🔷 Algorithm 10: ADABOOST
---

In [ ]:
# ============================================================
# ALGORITHM 10: ADABOOST
# ============================================================
print("🔷 ALGORITHM 10: ADABOOST")
print("="*60)

# Train AdaBoost Model
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=100, learning_rate=0.5, random_state=42
)
ada_model.fit(X_train, y_train)

# Predictions
y_pred_ada = ada_model.predict(X_test)
y_prob_ada = ada_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_ada = evaluate_classifier("AdaBoost", y_test, y_pred_ada, y_prob_ada)
all_results.append(result_ada)

In [ ]:
# AdaBoost - Feature Importance
feat_imp_ada = pd.DataFrame({'Feature': X.columns, 'Importance': ada_model.feature_importances_})
feat_imp_ada = feat_imp_ada.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, max(6, len(X.columns)*0.4)))
plt.barh(feat_imp_ada['Feature'], feat_imp_ada['Importance'], color='#1abc9c', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('AdaBoost - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🔷 Algorithm 11: K-MEANS CLUSTERING
---

In [ ]:
# ============================================================
# ALGORITHM 11: K-MEANS CLUSTERING
# ============================================================
print("🔷 ALGORITHM 11: K-MEANS CLUSTERING")
print("Note: K-Means is an unsupervised algorithm.")
print("="*60)

# Elbow Method to find optimal K
inertias = []
silhouette_scores_list = []
K_range = range(2, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels_temp = kmeans_temp.fit_predict(X_train_scaled)
    inertias.append(kmeans_temp.inertia_)
    sil_score = silhouette_score(X_train_scaled, labels_temp, sample_size=min(5000, len(X_train_scaled)))
    silhouette_scores_list.append(sil_score)
    print(f"K={k}: Inertia={kmeans_temp.inertia_:.2f}, Silhouette={sil_score:.4f}")

In [ ]:
# Elbow & Silhouette Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Plot
axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-Cluster SS)', fontsize=12)
axes[0].set_title('K-Means: Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette Score Plot
axes[1].plot(list(K_range), silhouette_scores_list, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('K-Means: Silhouette Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means with optimal K (using 2 for binary comparison with target)
optimal_k = 2  # For comparison with binary target
best_sil_k = list(K_range)[np.argmax(silhouette_scores_list)]
print(f"Best Silhouette K = {best_sil_k}")
print(f"Using K = {optimal_k} for comparison with target labels")

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels_train = kmeans_model.fit_predict(X_train_scaled)
kmeans_labels_test = kmeans_model.predict(X_test_scaled)

# Silhouette Score
sil_score_final = silhouette_score(X_test_scaled, kmeans_labels_test, 
                                    sample_size=min(5000, len(X_test_scaled)))
print(f"\nSilhouette Score (K={optimal_k}): {sil_score_final:.4f}")

# Compare clusters with actual labels
print(f"\nCluster Distribution (Test):")
print(pd.Series(kmeans_labels_test).value_counts())

# Cross-tabulation
print(f"\nCluster vs Actual Label Cross-tab:")
ct = pd.crosstab(y_test, kmeans_labels_test, rownames=['Actual'], colnames=['Cluster'])
print(ct)

In [ ]:
# K-Means Visualization using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cluster assignments
scatter1 = axes[0].scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=kmeans_labels_test, 
                           cmap='viridis', alpha=0.5, s=10)
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].set_title('K-Means Clusters (PCA Projection)', fontsize=12, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Actual labels
scatter2 = axes[1].scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, 
                           cmap='coolwarm', alpha=0.5, s=10)
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
axes[1].set_title('Actual Labels (PCA Projection)', fontsize=12, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Actual Label')

plt.tight_layout()
plt.show()

---
## 🔷 Algorithm 12: DBSCAN
---

In [ ]:
# ============================================================
# ALGORITHM 12: DBSCAN
# ============================================================
print("🔷 ALGORITHM 12: DBSCAN")
print("Note: DBSCAN is a density-based unsupervised clustering algorithm.")
print("="*60)

# Use a subset for DBSCAN if dataset is large
max_dbscan_samples = 5000
if len(X_test_scaled) > max_dbscan_samples:
    print(f"⚠️ Using subset of {max_dbscan_samples} samples for DBSCAN")
    dbscan_indices = np.random.RandomState(42).choice(len(X_test_scaled), max_dbscan_samples, replace=False)
    X_dbscan = X_test_scaled[dbscan_indices]
    y_dbscan_actual = y_test.iloc[dbscan_indices]
else:
    X_dbscan = X_test_scaled
    y_dbscan_actual = y_test

# Try different eps values
eps_values = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
print("\nTesting different eps values:")
print("-" * 50)

dbscan_results = []
for eps in eps_values:
    dbscan_temp = DBSCAN(eps=eps, min_samples=5)
    labels_temp = dbscan_temp.fit_predict(X_dbscan)
    n_clusters = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_noise = list(labels_temp).count(-1)
    noise_pct = n_noise / len(labels_temp) * 100
    
    sil = -1
    if n_clusters >= 2 and n_clusters < len(X_dbscan):
        non_noise_mask = labels_temp != -1
        if non_noise_mask.sum() > n_clusters:
            try:
                sil = silhouette_score(X_dbscan[non_noise_mask], labels_temp[non_noise_mask])
            except:
                sil = -1
    
    dbscan_results.append({'eps': eps, 'clusters': n_clusters, 'noise': n_noise, 
                           'noise_pct': noise_pct, 'silhouette': sil})
    print(f"eps={eps:.1f}: Clusters={n_clusters}, Noise={n_noise} ({noise_pct:.1f}%), Silhouette={sil:.4f}")

dbscan_results_df = pd.DataFrame(dbscan_results)

In [ ]:
# Apply DBSCAN with a reasonable eps
# Choose eps that gives a good balance
valid_results = [r for r in dbscan_results if r['clusters'] >= 2 and r['noise_pct'] < 50]
if valid_results:
    best_dbscan = max(valid_results, key=lambda x: x['silhouette'])
    best_eps = best_dbscan['eps']
else:
    best_eps = 2.0  # Default

print(f"\nSelected eps = {best_eps}")

dbscan_model = DBSCAN(eps=best_eps, min_samples=5)
dbscan_labels = dbscan_model.fit_predict(X_dbscan)

n_clusters_final = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_final = list(dbscan_labels).count(-1)

print(f"Number of Clusters: {n_clusters_final}")
print(f"Number of Noise Points: {n_noise_final} ({n_noise_final/len(dbscan_labels)*100:.1f}%)")
print(f"\nCluster Distribution:")
print(pd.Series(dbscan_labels).value_counts().sort_index())

In [ ]:
# DBSCAN Visualization
X_dbscan_pca = pca.transform(X_dbscan)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DBSCAN clusters
unique_labels = set(dbscan_labels)
colors_db = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(sorted(unique_labels), colors_db):
    mask = dbscan_labels == label
    if label == -1:
        axes[0].scatter(X_dbscan_pca[mask, 0], X_dbscan_pca[mask, 1], 
                       c='gray', marker='x', s=15, alpha=0.3, label='Noise')
    else:
        axes[0].scatter(X_dbscan_pca[mask, 0], X_dbscan_pca[mask, 1], 
                       c=[color], s=10, alpha=0.5, label=f'Cluster {label}')

axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].set_title(f'DBSCAN Clusters (eps={best_eps})', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='best')
axes[0].grid(True, alpha=0.3)

# Actual labels comparison
scatter_db = axes[1].scatter(X_dbscan_pca[:, 0], X_dbscan_pca[:, 1], 
                             c=y_dbscan_actual.values, cmap='coolwarm', alpha=0.5, s=10)
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
axes[1].set_title('Actual Labels (PCA Projection)', fontsize=12, fontweight='bold')
plt.colorbar(scatter_db, ax=axes[1], label='Actual Label')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cross-tabulation
if n_clusters_final >= 1:
    print("\nCluster vs Actual Label Cross-tab:")
    ct_dbscan = pd.crosstab(y_dbscan_actual, dbscan_labels, 
                            rownames=['Actual'], colnames=['DBSCAN Cluster'])
    print(ct_dbscan)

---
## 🔷 Algorithm 13: ENSEMBLE LEARNING
---

In [ ]:
# ============================================================
# ALGORITHM 13: ENSEMBLE LEARNING
# ============================================================
print("🔷 ALGORITHM 13: ENSEMBLE LEARNING")
print("Implementing: Voting Classifier, Bagging, and Stacking")
print("="*60)

In [ ]:
# ---- 13a: VOTING CLASSIFIER (Hard & Soft) ----
print("\n" + "="*60)
print("🔹 13a: VOTING CLASSIFIER (Soft Voting)")
print("="*60)

# Create individual estimators
estimators_voting = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=50, random_state=42, use_label_encoder=False, 
                          eval_metric='logloss', verbosity=0))
]

# Soft Voting (uses predict_proba)
voting_soft = VotingClassifier(estimators=estimators_voting, voting='soft', n_jobs=-1)
voting_soft.fit(X_train_scaled, y_train)

# Predictions
y_pred_vote_soft = voting_soft.predict(X_test_scaled)
y_prob_vote_soft = voting_soft.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

# Evaluate
result_vote_soft = evaluate_classifier("Voting Classifier (Soft)", y_test, 
                                        y_pred_vote_soft, y_prob_vote_soft)
all_results.append(result_vote_soft)

In [ ]:
# ---- 13b: HARD VOTING ----
print("\n" + "="*60)
print("🔹 13b: VOTING CLASSIFIER (Hard Voting)")
print("="*60)

voting_hard = VotingClassifier(estimators=estimators_voting, voting='hard', n_jobs=-1)
voting_hard.fit(X_train_scaled, y_train)

y_pred_vote_hard = voting_hard.predict(X_test_scaled)

result_vote_hard = evaluate_classifier("Voting Classifier (Hard)", y_test, y_pred_vote_hard)
all_results.append(result_vote_hard)

In [ ]:
# ---- 13c: BAGGING CLASSIFIER ----
print("\n" + "="*60)
print("🔹 13c: BAGGING CLASSIFIER")
print("="*60)

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=10),
    n_estimators=50, random_state=42, n_jobs=-1
)
bagging_model.fit(X_train, y_train)

y_pred_bag = bagging_model.predict(X_test)
y_prob_bag = bagging_model.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

result_bag = evaluate_classifier("Bagging Classifier", y_test, y_pred_bag, y_prob_bag)
all_results.append(result_bag)

In [ ]:
# ---- 13d: STACKING CLASSIFIER ----
print("\n" + "="*60)
print("🔹 13d: STACKING CLASSIFIER")
print("="*60)

# Base estimators
estimators_stack = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1))
]

# Meta-learner
stacking_model = StackingClassifier(
    estimators=estimators_stack,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5, n_jobs=-1
)
stacking_model.fit(X_train_scaled, y_train)

y_pred_stack = stacking_model.predict(X_test_scaled)
y_prob_stack = stacking_model.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y)) == 2 else None

result_stack = evaluate_classifier("Stacking Classifier", y_test, y_pred_stack, y_prob_stack)
all_results.append(result_stack)

---
## 📊 FINAL COMPARISON OF ALL ALGORITHMS
---

In [ ]:
# ============================================================
# FINAL COMPARISON TABLE
# ============================================================
print("\n" + "="*80)
print("📊 FINAL COMPARISON OF ALL 13 ALGORITHMS")
print("="*80)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1  # Start index from 1

# Format percentages
results_display = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    results_display[col] = results_display[col].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("\n")
print(results_display.to_string())
print("\n")

# Highlight best model
best_model = results_df.iloc[0]
print(f"🏆 BEST MODEL: {best_model['Model']}")
print(f"   Accuracy:  {best_model['Accuracy']:.4f} ({best_model['Accuracy']*100:.2f}%)")
print(f"   Precision: {best_model['Precision']:.4f}")
print(f"   Recall:    {best_model['Recall']:.4f}")
print(f"   F1-Score:  {best_model['F1-Score']:.4f}")

In [ ]:
# ============================================================
# FINAL COMPARISON VISUALIZATIONS
# ============================================================

# 1. Accuracy Comparison Bar Chart
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Sort by accuracy for plotting
results_sorted = results_df.sort_values('Accuracy', ascending=True)

# Accuracy
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(results_sorted)))
axes[0, 0].barh(results_sorted['Model'], results_sorted['Accuracy'], color=colors, edgecolor='black')
axes[0, 0].set_xlabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Model Comparison - Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim([max(0, results_sorted['Accuracy'].min() - 0.05), 1.0])
for i, v in enumerate(results_sorted['Accuracy']):
    axes[0, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Precision
axes[0, 1].barh(results_sorted['Model'], results_sorted['Precision'], color=colors, edgecolor='black')
axes[0, 1].set_xlabel('Precision', fontsize=12)
axes[0, 1].set_title('Model Comparison - Precision', fontsize=14, fontweight='bold')
axes[0, 1].set_xlim([max(0, results_sorted['Precision'].min() - 0.05), 1.0])
for i, v in enumerate(results_sorted['Precision']):
    axes[0, 1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Recall
axes[1, 0].barh(results_sorted['Model'], results_sorted['Recall'], color=colors, edgecolor='black')
axes[1, 0].set_xlabel('Recall', fontsize=12)
axes[1, 0].set_title('Model Comparison - Recall', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim([max(0, results_sorted['Recall'].min() - 0.05), 1.0])
for i, v in enumerate(results_sorted['Recall']):
    axes[1, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# F1-Score
axes[1, 1].barh(results_sorted['Model'], results_sorted['F1-Score'], color=colors, edgecolor='black')
axes[1, 1].set_xlabel('F1-Score', fontsize=12)
axes[1, 1].set_title('Model Comparison - F1-Score', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim([max(0, results_sorted['F1-Score'].min() - 0.05), 1.0])
for i, v in enumerate(results_sorted['F1-Score']):
    axes[1, 1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.suptitle('📊 All 13 Algorithms - Performance Comparison', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Grouped Bar Chart
fig, ax = plt.subplots(figsize=(18, 8))

models = results_df['Model']
x = np.arange(len(models))
width = 0.2

bars1 = ax.bar(x - 1.5*width, results_df['Accuracy'], width, label='Accuracy', 
               color='#3498db', edgecolor='black')
bars2 = ax.bar(x - 0.5*width, results_df['Precision'], width, label='Precision', 
               color='#2ecc71', edgecolor='black')
bars3 = ax.bar(x + 0.5*width, results_df['Recall'], width, label='Recall', 
               color='#e74c3c', edgecolor='black')
bars4 = ax.bar(x + 1.5*width, results_df['F1-Score'], width, label='F1-Score', 
               color='#f39c12', edgecolor='black')

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('All Algorithms - Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

In [ ]:
# 3. Radar Chart for Top 5 Models
top_n = min(5, len(results_df))
top_models = results_df.head(top_n)

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
num_vars = len(categories)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

colors_radar = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, (_, row) in enumerate(top_models.iterrows()):
    values = [row[cat] for cat in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors_radar[i])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title(f'Top {top_n} Models - Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Heatmap of all metrics
plt.figure(figsize=(12, max(6, len(results_df)*0.5)))
heatmap_data = results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn', 
            linewidths=0.5, vmin=0, vmax=1, center=0.5)
plt.title('All Algorithms - Performance Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*80)
print("📋 FINAL SUMMARY")
print("="*80)
print(f"\n📂 Dataset: loan_default_dataset.csv")
print(f"📊 Total Samples: {len(df)}")
print(f"🎯 Target Column: {target_col}")
print(f"📐 Features Used: {X.shape[1]}")
print(f"📏 Training Samples: {len(X_train)}")
print(f"📏 Testing Samples: {len(X_test)}")
print(f"\n🤖 Total Algorithms Evaluated: {len(all_results)}")
print("\n" + "-"*80)

print("\n🏆 RANKING (by Accuracy):")
print("-"*80)
for i, (_, row) in enumerate(results_df.iterrows(), 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f" {i}."
    print(f"  {medal} {row['Model']:<35} Acc: {row['Accuracy']:.4f}  F1: {row['F1-Score']:.4f}")

print("\n" + "="*80)
print("\n✅ Analysis Complete! All 13 algorithms have been evaluated.")
print("\n📌 Note: K-Means & DBSCAN are unsupervised algorithms and were")
print("   evaluated separately using clustering metrics (Silhouette Score).")
print("="*80)